## 生成数据集

In [3]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [4]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## 读取数据集

我们将features和labels作为API的参数传递，并通过数据迭代器指定batch_size。 此外，布尔值is_train表示是否希望数据迭代器对象在每个迭代周期内打乱数据。

In [5]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [6]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [7]:
next(iter(data_iter))

[tensor([[ 0.5732,  0.4979],
         [ 0.1170, -1.8189],
         [-0.2806,  0.3031],
         [-1.4910,  1.2618],
         [ 0.1027,  1.2133],
         [-0.8443,  1.0961],
         [-0.2181, -0.5676],
         [-0.1014,  1.5462],
         [-0.1346,  2.2583],
         [ 1.1390,  0.1068]]),
 tensor([[ 3.6718],
         [10.6261],
         [ 2.6070],
         [-3.0703],
         [ 0.2730],
         [-1.2058],
         [ 5.7027],
         [-1.2432],
         [-3.7329],
         [ 6.1375]])]

## 定义模型

在PyTorch中，全连接层在`Linear`类中定义，值得注意的是，我们将两个参数传递到`nn.Linear`中，第一个指定输入特征形状，即2，第二个指定输出特征形状，输出特征形状为单个标量，因此为1。

In [ ]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

 (**初始化模型参数**)

在使用`net`之前，我们需要初始化模型参数。
如在线性回归模型中的权重和偏置。
深度学习框架通常有预定义的方法来初始化参数。
在这里，我们指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样，
偏置参数将初始化为零。


In [ ]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

## 定义损失函数

In [ ]:
loss = nn.MSELoss()

## 定义优化算法

In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练

在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），不停地从中获取一个小批量的输入和相应的标签，对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）  
* 通过进行反向传播来计算梯度  
* 通过调用优化器来更新模型参数  

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。

In [ ]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

**比较生成数据集的真实参数和通过有限数据训练获得的模型参数**

要访问参数，我们首先从`net`访问所需的层，然后读取该层的权重和偏置，正如在从零开始实现中一样，我们估计得到的参数与生成数据的真实参数非常接近。

In [ ]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)